## **FarmerMitra**

### **Step 1: Download, Filter, and Check Dataset Health**
We download the data and count the samples to ensure the dataset is balanced and 'healthy'.

In [ ]:
import kagglehub
import os
import shutil
import matplotlib.pyplot as plt

# --- Download Datasets ---
path_pv = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
path_pd = kagglehub.dataset_download("abdulhasibuddin/plant-doc-dataset")
path_new_dataset = kagglehub.dataset_download("hadyahmed00/plants-leafs-dataset") # Added new dataset

print("Path to PlantVillage dataset files:", path_pv)
print("Path to PlantDoc dataset files:", path_pd)
print("Path to new dataset files:", path_new_dataset)

In [ ]:
target_dir = "./filtered_dataset"
classes_to_keep = ["Apple", "Grape", "Tomato"]
if os.path.exists(target_dir): shutil.rmtree(target_dir)
os.makedirs(target_dir, exist_ok=True)

# Helper to merge folders
def merge_into_target(source_root, keywords):
    if not os.path.exists(source_root):
        print(f"Warning: {source_root} does not exist. Skipping merge for this source.")
        return
    for folder in os.listdir(source_root):
        # Check if any keyword is a substring of the folder name (case-insensitive)
        if any(k.lower() in folder.lower() for k in keywords):
            src = os.path.join(source_root, folder)
            dst = os.path.join(target_dir, folder)
            if os.path.isdir(src):
                if os.path.exists(dst):
                    # If destination folder exists, merge contents
                    for item_name in os.listdir(src):
                        s = os.path.join(src, item_name)
                        d = os.path.join(dst, item_name)
                        if os.path.isdir(s):
                            shutil.copytree(s, d, dirs_exist_ok=True)
                        else:
                            shutil.copy2(s, d)
                else:
                    shutil.copytree(src, dst)


# Merge PlantVillage
pv_root = os.path.join(path_pv, "plantvillage dataset", "color")
merge_into_target(pv_root, classes_to_keep)

# Merge PlantDoc (using the path provided by the user)
pd_root = os.path.join(path_pd, "PlantDoc-Dataset", "train") # Corrected path based on user input
merge_into_target(pd_root, classes_to_keep)

# Merge new dataset (based on user feedback, assuming data is under 'Full/train')
new_dataset_root = os.path.join(path_new_dataset, "Full", "train") # Corrected path based on user input
merge_into_target(new_dataset_root, classes_to_keep)

# Health Check
class_counts = {d: len(os.listdir(os.path.join(target_dir, d))) for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))}
plt.figure(figsize=(10, 5))
plt.barh(list(class_counts.keys()), list(class_counts.values()))
plt.title("Multi-Source Dataset Health: Image Count per Class")
plt.show()
print(f"Total merged images: {sum(class_counts.values())}")

### **Step 2: Data Loaders and Transformations**
Preparing the images for the neural network.

In [ ]:
import torch
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset

# --- FIX 1: MapDataset Wrapper for independent transforms ---
class MapDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.dataset[index]
        if self.transform: x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.dataset)

# --- IMPROVEMENT: Strong Augmentations for Domain Gap ---
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

full_dataset = datasets.ImageFolder(target_dir)
train_indices, val_indices = random_split(range(len(full_dataset)), [0.8, 0.2], generator=torch.Generator().manual_seed(42))

train_data = MapDataset(Subset(full_dataset, train_indices), transform=data_transforms['train'])
val_data = MapDataset(Subset(full_dataset, val_indices), transform=data_transforms['val'])

# --- Class Weighting for Imbalance ---
labels = [full_dataset.targets[i] for i in train_indices]
class_counts = np.bincount(labels)
weights = 1. / torch.tensor(class_counts, dtype=torch.float)
sample_weights = weights[labels]
sampler = torch.utils.data.WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_data, batch_size=32, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, num_workers=2)

class_names = full_dataset.classes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dataset ready. Weighted sampler enabled. Using {device}")

### **Step 3: Model Initialization**
Loading EfficientNet-B4 and attaching a custom classifier.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import torch.optim as optim

# --- CBAM Attention Module ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )
        self.conv_after_concat = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x):
        # Channel Attention
        avg_out = self.fc(self.avg_pool(x).view(x.size(0), -1)).view(x.size(0), x.size(1), 1, 1)
        max_out = self.fc(self.max_pool(x).view(x.size(0), -1)).view(x.size(0), x.size(1), 1, 1)
        x = x * torch.sigmoid(avg_out + max_out)
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial = self.conv_after_concat(torch.cat([avg_out, max_out], dim=1))
        return x * torch.sigmoid(spatial)

# --- Model with ConvNeXt + CBAM ---
base_model = models.convnext_tiny(weights='IMAGENET1K_V1')

class CustomConvNext(nn.Module):
    def __init__(self, base, num_classes):
        super().__init__()
        self.features = base.features
        self.attention = CBAM(768) # ConvNeXt-Tiny last channel is 768
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.attention(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = CustomConvNext(base_model, len(class_names)).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(weight=weights.to(device))
print("Model initialized: ConvNeXt-Tiny + CBAM Attention Layer.")

### **Step 4: Training Loop**
Executing the training process.

In [ ]:
import torch
import torch.optim as optim
import copy
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score # Added imports

def train_model(model, criterion, optimizer, num_epochs=10):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    train_loss_history = []
    val_loss_history = []
    train_acc_history = []
    val_acc_history = []
    train_f1_history = []
    val_f1_history = []
    train_precision_history = []
    val_precision_history = []
    train_recall_history = []
    val_recall_history = []

    # Learning rate scheduler for better convergence
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=2, factor=0.5)

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
                dataset_size = len(train_data)
            else:
                model.eval()
                dataloader = val_loader
                dataset_size = len(val_data)

            running_loss, running_corrects = 0.0, 0
            all_labels_phase = [] # To store labels for F1, precision, recall
            all_preds_phase = []  # To store predictions for F1, precision, recall

            # tqdm progress bar
            pbar = tqdm(enumerate(dataloader),
                        total=len(dataloader),
                        desc=f"{phase.capitalize()}")

            for i, (inputs, labels) in pbar:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                all_labels_phase.extend(labels.cpu().numpy())
                all_preds_phase.extend(preds.cpu().numpy())

                pbar.set_postfix({'loss': loss.item()})

            epoch_loss = running_loss / dataset_size
            epoch_acc = running_corrects.double() / dataset_size

            # Calculate F1, Precision, Recall for the current phase/epoch
            if len(all_labels_phase) > 0: # Ensure there's data to calculate metrics
                epoch_f1 = f1_score(all_labels_phase, all_preds_phase, average='weighted', zero_division=0)
                epoch_precision = precision_score(all_labels_phase, all_preds_phase, average='weighted', zero_division=0)
                epoch_recall = recall_score(all_labels_phase, all_preds_phase, average='weighted', zero_division=0)
            else:
                epoch_f1, epoch_precision, epoch_recall = 0.0, 0.0, 0.0 # Handle empty case

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} F1: {epoch_f1:.4f} Precision: {epoch_precision:.4f} Recall: {epoch_recall:.4f}')

            if phase == 'train':
                train_loss_history.append(epoch_loss)
                train_acc_history.append(epoch_acc.item()) # .item() to get scalar from tensor
                train_f1_history.append(epoch_f1)
                train_precision_history.append(epoch_precision)
                train_recall_history.append(epoch_recall)
            else: # phase == 'val'
                val_loss_history.append(epoch_loss)
                val_acc_history.append(epoch_acc.item()) # .item() to get scalar from tensor
                val_f1_history.append(epoch_f1)
                val_precision_history.append(epoch_precision)
                val_recall_history.append(epoch_recall)

                scheduler.step(epoch_acc)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
        print()

    model.load_state_dict(best_model_wts)
    return model, {
        'train_loss': train_loss_history, 'val_loss': val_loss_history,
        'train_acc': train_acc_history, 'val_acc': val_acc_history,
        'train_f1': train_f1_history, 'val_f1': val_f1_history,
        'train_precision': train_precision_history, 'val_precision': val_precision_history,
        'train_recall': train_recall_history, 'val_recall': val_recall_history
    }

# Execute Training
trained_model, history = train_model(model, criterion, optimizer, num_epochs=10)

In [ ]:
import torch
from google.colab import files

# Define the filename for the weights
weights_filename = 'farmer-mitra_weights.pth'

# Save the state dictionary of the trained model
torch.save(trained_model.state_dict(), weights_filename)

print(f"Model weights saved to {weights_filename}.")

# Provide a link to download the file
files.download(weights_filename)

print("A download prompt should appear in your browser. If not, check your browser's download settings.")

Model weights saved to farmer-mitra_weights.pth.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

A download prompt should appear in your browser. If not, check your browser's download settings.


In [ ]:
from google.colab import files
from PIL import Image
import io
import torch.nn.functional as F
import matplotlib.pyplot as plt

def upload_and_predict():
    uploaded = files.upload()

    for filename in uploaded.keys():
        # Load user image
        img = Image.open(io.BytesIO(uploaded[filename])).convert('RGB')

        # Preprocess
        img_t = data_transforms['val'](img).unsqueeze(0).to(device)

        # Predict
        trained_model.eval()
        with torch.no_grad():
            outputs = trained_model(img_t)
            probs = F.softmax(outputs, dim=1)
            conf, preds = torch.max(probs, 1)

        # Display Results
        print(f"\n--- User Upload Results ---")
        print(f"File: {filename}")
        print(f"Model Prediction: {class_names[preds[0]]}")
        print(f"Confidence: {conf.item():.2%}")

        plt.imshow(img)
        plt.title(f"Prediction: {class_names[preds[0]]}")
        plt.axis('off')
        plt.show()

upload_and_predict()

In [ ]:
import torch
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure the model is in evaluation mode
trained_model.eval()

all_labels = []
all_preds = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = trained_model(inputs)
        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

# Convert lists to numpy arrays
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

# Calculate F1 Score
f1 = f1_score(all_labels, all_preds, average='weighted') # 'weighted' accounts for class imbalance
print(f"F1 Score (weighted): {f1:.4f}")

# Calculate Precision
precision = precision_score(all_labels, all_preds, average='weighted')
print(f"Precision (weighted): {precision:.4f}")

# Calculate Recall
recall = recall_score(all_labels, all_preds, average='weighted')
print(f"Recall (weighted): {recall:.4f}")

# Generate Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot Confusion Matrix
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()